## Beaty Museum Informatics x MDS Capstone: Project Proposal
#### May 8, 2026
Authors: Molly Kessler, Wendy Frankel, William Song, Johnson Chuang

Project Mentor: Payman Nickhi, UBC MDS Faculty

Capstone Partner: Paul Bucci, Informatics Curator at Beaty Biodiversity Museum

### Executive Summary

At the Beaty Biodiversity Museum, curators keep millions of physical specimen organized and labeled based on the most current scientific consensus about their taxonomy, including the species name and phylogenetic tree. Since taxonomy is constantly changing, many different synonyms may be used to label the same species in different publications. We propose a web app that synthesizes taxonomic information from many online sources, providing a interface for curators to easily compare competing taxonomic information to decide which designation to incorporate into the museum's database.

### Introduction

Taxonomy refers to classifying organisms into hierarchical groups based on shared characteristics and evolutionary relationships. Within Beaty, curators are responsible for ensuring every specimen is identified, classified, and stored appropriately. Since taxonomy is an ever evolving field, individual species names can change over time, introducing synonyms, basionyms, variants, subspecies, and more. Beyond just the name, entire taxonomic classifications (e.g. family, order) can change, introducing further uncertainty around where organisms should physically and digitally be stored.


Many online databases compile updated species names and taxonomic classification changes from millions of individual publications. Currently, curators access these resources individually, manually comparing competing information based on their expert knowledge of Beaty's current organization scheme. Our web app will speed up this process by making API calls to online sources and creating a combined list of species synonyms and additional information — including classification, range, date of discovery, and more — to help curators decide which synonym to use. This will give curators a more consistent, reproducible, and faster workflow to classify and label specimens.

### Data & Exploratory Data Analysis (EDA)

Data sources vary by taxonomic group, with some sources containing information for all taxa (e.g. GBIF, GenBank) and others being more specific to a particular group (e.g. Mushroom Observer for fungi). API calls return JSON-formatted information about a species. In particular, we want to access information related to species names and taxonomy, such as species name synonyms, variants, and subspecies. The size of data varies by species — a species might have no synonyms or dozens, and any given synonym might have many variants, subspecies, etc.


In [6]:
from scripts.APIs.GBIF import get_gbif_synonyms
from scripts.APIs.IndexFungorum import get_indexfungorum_synonyms

species = "Amanita muscaria"

gbif_synonyms = set(get_gbif_synonyms(species).keys())
if_synonyms = set(get_indexfungorum_synonyms(species).keys())

all_synonyms = gbif_synonyms | if_synonyms

df = pd.DataFrame([
    {
        "Synonym": name,
        "Sources": ", ".join([
            s for s, sset in [("GBIF", gbif_synonyms), ("Index Fungorum", if_synonyms)]
            if name in sset
        ])
    }
    for name in sorted(all_synonyms)
])
df

,Synonym,Sources
0,Agaricus aureolus,"GBIF, Index Fungorum"
1,Agaricus imperialis,"GBIF, Index Fungorum"
2,Agaricus muscarius,"GBIF, Index Fungorum"
3,Agaricus nobilis,"GBIF, Index Fungorum"
4,Agaricus pseudoaurantiacus,"GBIF, Index Fungorum"
5,Agaricus puellus,"GBIF, Index Fungorum"
6,Amanita aureola,"GBIF, Index Fungorum"
7,Amanita circinnata,"GBIF, Index Fungorum"
8,Amanita formosa,"GBIF, Index Fungorum"
9,Amanita muscaria,"GBIF, Index Fungorum"


Table 1. Species *Amanita muscaria* synonyms from GBIF and Index Fungorum. 

A main challenge when working with this data is that performing many API calls will become very slow, and we will need to optimize our code and likely implement caching to speed up searches. Additionally, the output format will across sources, requiring substantial data cleaning and post-processing to collate data into a consistent format.

### Data Science Approach

Our data pipeline will automate the collection of records from online sources, displaying them in a visual interface for curators to make faster and more informed decisions. We will provide search results from authoritiative sources for all kinds of taxa (i.e. fungi, insects, etc) in a consistent format by implementing API calls to each source and unifying the output format. Each source will have its own script responsible for retrieving relevant records and cleaning them into the consistent format. To ensure consistency as the codebase grows, we will consider defining a Python abstract class with database-specific subclasses so that future developers can easily add new sources that conform to the same contract.
  
A key part of what we are building is not just data retrieval, but also incorporating provenance and context. Authorities change over time as data sources become stale, funding lapses, and scientific consensus shifts. Curators don't just need a list of synonyms — they need to know what sources list a name, when it was published, and how widely it is recognized across different scientific communities. For example, a well established synonym that is primarily used in South America may be less relevant for Beaty than a less common synonym that is most used in Canada. Synonyms will be accompanied metrics such as coverage (how many databases list the name), recency (how recently the name was published/became popular), specificity (whether the name is commmon in more general vs. taxa-specific resources), and more, giving curators a quick sense of which names are recognized and current.
 
We will explore a variety of interface designs to present the reconciled information to curators, including a comparison table, a node-based expandable graph, and an LLM summary view summarized view powered by a Retrieval-Augmented Generation (RAG) approach. In all designs, our priority is to surface information without introducing bias, and always to provide information to curators, not to make any automated decisions. Aggregated summaries in particular carry a risk of obscuring disagreement between sources, so we will be thoughtful about how we present confidence and provenance alongside any LLM summary. 

The final deliverable will be a working pipeline and app hosted on Beaty's GitHub repository, with documentation that allows curators and future developers to use and extend it. A curator will be able to access the app at a public URl and use it to search a species name and retrieve a reconciled view of relevant records from all configured sources, along with provenance metadata for each result.

### Evaluation & Success Criteria

We define success in terms of the reliability and completeness of the information we surface, and whether that information helps curators make faster decisions and feel more informed. Our primary success condition is that the pipeline can accurately retrieve, normalize, and compare records from the configured data sources, and present them to curators in a way that is clear, transparent, and free of systematic bias.

We will evaluate success through feedback from curators at Beaty. Their ability to use the tool effectively, and their confidence in the provenance and authority of the information it surfaces, is our most important qualitative measure of success. If curators can understand where each piece of data came from and feel the tool has improved their ability to make informed determinations, we will have succeeded. In order to achieve success for curators working across many different collections, we will need to implement many different sources. Currently we expect to implement over 30 individual sources into our pipeline, but this number may grow as we encourage curators to request the sources they use.

We will be extremely attentive to evaluating the LLM component for bias, hallucinations, and appropriate level of detail. We will review LLM outputs across a range of taxa and source combinations to check whether the model consistently favors certain databases, naming conventions, or geographic authorities without justification from the underlying data. If we identify issues, we will attempt to address them through prompt engineering or model selection, and if unsuccessful, will carefully consider whether it is safe and ethical to include an LLM at this time.

### Timeline

**Before May 8**: Present proposal and initial prototypes to mentor and capstone partner for feedback.

**May 15 – 22**: Develop preliminary search system capable of returning species names collated from 10+ API sources. Gather curator feedback on working and paper prototypes.

**By May 29**: Add additional species details and metrics (coverage, recency, etc.) for each synonym. Complete initial LLM development and begin evaluation. Continue adding APIs.

**June 8**: Finish implementing 30+ APIs. Complete initial model evaluation. Submit a runnable draft to mentor for review. 

**June 8 – 10**: Finalize presentation slides.

**June 24 – 25**: Deliver final version of report and data product to capstone partner. Present final product.
